# exp-015: Ablation Study — fusion 6요소 단일 제거

- **목적:** Model C의 6요소(role_semantic / hard_skill / competency / achievement / industry / quality_adjustment) 중 어느 것을 빼면 NDCG@10이 가장 떨어지나 정량화.
- **차별성 축:** ① 자소서 정성 + ④ GT 부재 (논문 §Ablation 표 산출)
- **방법:** Leave-one-out — 각 요소 i를 빼고(가중치 0) 나머지를 *비율 유지*하며 재정규화 → 5관점 NDCG@10 측정
- **데이터:** 
  - `weighted_results.csv` 6요소 + `benchmark_labeled_100_{A,B,C,D,E}.csv` 라벨
  - SINGLE 가중치 기준 (`role 0.35 / skill 0.20 / comp 0.15 / achv 0.10 / ind 0.10 / qual 0.10`)
- **관련 위키:** 관점별-fusion-가중치-설계 §1.1, dual-encoder-진화-실험계획 exp-015
- **작성일/실행일:** 2026-05-18
- **풀데이터 필요?** ❌ 불필요

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json

DATA = Path('raw/data/gemini_profile_outputs')
OUT_DIR = Path('raw/experiments/exp-015-ablation')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 데이터 로드
labels = {}
for p in 'ABCDE':
    df = pd.read_csv(DATA / f'benchmark_labeled_100_{p}.csv', encoding='utf-8-sig')
    df.columns = [c.lstrip('\ufeff') for c in df.columns]
    df['job_id'] = df['job_id'].astype(str)
    labels[p] = df

wr = pd.read_csv(DATA / 'weighted_results.csv', encoding='utf-8-sig')
wr.columns = [c.lstrip('\ufeff') for c in wr.columns]
wr['job_id'] = wr['job_id'].astype(str)
fusion_cols = ['role_semantic', 'hard_skill', 'competency', 'achievement', 'industry', 'quality_adjustment']

# SINGLE 기준 가중치
SINGLE_WEIGHTS = {
    'role_semantic': 0.35, 'hard_skill': 0.20, 'competency': 0.15,
    'achievement': 0.10, 'industry': 0.10, 'quality_adjustment': 0.10,
}
print(f'SINGLE 합: {sum(SINGLE_WEIGHTS.values())}')

SINGLE 합: 1.0


In [2]:
# Ablation 가중치 세트 생성
# Strategy 1: 해당 요소 0, 나머지 합=1로 정규화 (비율 유지)
def ablation_weights(base, drop_col):
    w = {k: 0 if k == drop_col else v for k, v in base.items()}
    total = sum(w.values())
    return {k: v/total for k, v in w.items()}

ablations = {'FULL': SINGLE_WEIGHTS}
for col in fusion_cols:
    ablations[f'-{col}'] = ablation_weights(SINGLE_WEIGHTS, col)

# 가중치 확인
ab_df = pd.DataFrame(ablations).T
ab_df['sum'] = ab_df.sum(axis=1)
print('=== Ablation 가중치 세트 ===')
print(ab_df.round(4))

=== Ablation 가중치 세트 ===
                     role_semantic  hard_skill  competency  achievement  \
FULL                        0.3500      0.2000      0.1500       0.1000   
-role_semantic              0.0000      0.3077      0.2308       0.1538   
-hard_skill                 0.4375      0.0000      0.1875       0.1250   
-competency                 0.4118      0.2353      0.0000       0.1176   
-achievement                0.3889      0.2222      0.1667       0.0000   
-industry                   0.3889      0.2222      0.1667       0.1111   
-quality_adjustment         0.3889      0.2222      0.1667       0.1111   

                     industry  quality_adjustment  sum  
FULL                   0.1000              0.1000  1.0  
-role_semantic         0.1538              0.1538  1.0  
-hard_skill            0.1250              0.1250  1.0  
-competency            0.1176              0.1176  1.0  
-achievement           0.1111              0.1111  1.0  
-industry              0.0000    

In [3]:
# NDCG@K
def ndcg_at_k(rels, k=10):
    rels = np.asarray(rels, dtype=float)
    if len(rels) == 0: return 0.0
    rels_k = rels[:k]
    gains = (2**rels_k - 1) / np.log2(np.arange(2, len(rels_k)+2))
    ideal = np.sort(rels)[::-1][:k]
    ideal_gains = (2**ideal - 1) / np.log2(np.arange(2, len(ideal)+2))
    return gains.sum() / ideal_gains.sum() if ideal_gains.sum() > 0 else 0.0

# Ablation × 5 관점 NDCG 계산
results = []
for ab_name, weights in ablations.items():
    for eval_code in 'ABCDE':
        L = labels[eval_code]
        M = L.merge(wr[['userId', 'job_id'] + fusion_cols], on=['userId', 'job_id'], how='left', suffixes=('_label', ''))
        for c in fusion_cols:
            if c in M.columns: M[c] = M[c].fillna(0)
            else: M[c] = 0
        M['score'] = sum(M[c] * weights[c] for c in fusion_cols)
        ndcgs10, ndcgs5 = [], []
        for uid, g in M.groupby('userId'):
            g_sorted = g.sort_values('score', ascending=False)
            rels = g_sorted['judge_relevance'].fillna(0).values
            ndcgs10.append(ndcg_at_k(rels, 10))
            ndcgs5.append(ndcg_at_k(rels, 5))
        results.append({
            'ablation': ab_name,
            'eval_dataset': eval_code,
            'NDCG@10': float(np.mean(ndcgs10)),
            'NDCG@5': float(np.mean(ndcgs5)),
        })

res_df = pd.DataFrame(results)
ndcg10_piv = res_df.pivot(index='ablation', columns='eval_dataset', values='NDCG@10')
ndcg10_piv = ndcg10_piv.loc[['FULL'] + [f'-{c}' for c in fusion_cols], ['A','B','C','D','E']]
ndcg10_piv['mean'] = ndcg10_piv.mean(axis=1)
print('=== Ablation NDCG@10 (관점별 + 평균) ===')
print(ndcg10_piv.round(4).to_string())
ndcg10_piv.to_csv(OUT_DIR / 'ablation_ndcg10.csv')

=== Ablation NDCG@10 (관점별 + 평균) ===
eval_dataset              A       B       C       D       E    mean
ablation                                                           
FULL                 0.9794  0.9367  0.9227  0.9564  0.9614  0.9513
-role_semantic       0.9794  0.9352  0.9227  0.9564  0.9614  0.9510
-hard_skill          0.9794  0.9352  0.9227  0.9703  0.9660  0.9547
-competency          0.9776  0.9318  0.9307  0.9564  0.9622  0.9517
-achievement         0.9794  0.9352  0.9227  0.9703  0.9532  0.9522
-industry            0.9691  0.9378  0.9279  0.9508  0.9556  0.9482
-quality_adjustment  0.9776  0.9287  0.9227  0.9564  0.9622  0.9495


In [4]:
# 각 요소의 기여도 = FULL - (-요소 제거)
contrib = pd.DataFrame()
full_row = ndcg10_piv.loc['FULL']
for col in fusion_cols:
    drop_row = ndcg10_piv.loc[f'-{col}']
    contrib[col] = full_row - drop_row

contrib = contrib.T
contrib['mean_drop'] = contrib['mean']
contrib = contrib.sort_values('mean_drop', ascending=False)
print('=== 요소별 기여도 (제거 시 NDCG@10 하락폭) ===')
print('양수=중요(빼면 NDCG↓), 음수=노이즈(빼면 NDCG↑)')
print(contrib.round(4).to_string())
contrib.to_csv(OUT_DIR / 'feature_contribution.csv')

=== 요소별 기여도 (제거 시 NDCG@10 하락폭) ===
양수=중요(빼면 NDCG↓), 음수=노이즈(빼면 NDCG↑)
eval_dataset             A       B       C       D       E    mean  mean_drop
industry            0.0102 -0.0010 -0.0052  0.0056  0.0058  0.0031     0.0031
quality_adjustment  0.0017  0.0080  0.0000  0.0000 -0.0008  0.0018     0.0018
role_semantic       0.0000  0.0015  0.0000  0.0000  0.0000  0.0003     0.0003
competency          0.0017  0.0049 -0.0081  0.0000 -0.0008 -0.0004    -0.0004
achievement         0.0000  0.0015  0.0000 -0.0139  0.0082 -0.0009    -0.0009
hard_skill          0.0000  0.0015  0.0000 -0.0139 -0.0046 -0.0034    -0.0034


In [5]:
# 관점별 가장 중요한 요소 식별
print('=== 관점별 가장 큰 NDCG@10 기여 요소 (제거 시 가장 큰 하락) ===')
for p in 'ABCDE':
    drops = {col: full_row[p] - ndcg10_piv.loc[f'-{col}', p] for col in fusion_cols}
    sorted_drops = sorted(drops.items(), key=lambda x: x[1], reverse=True)
    top1, top2 = sorted_drops[0], sorted_drops[1]
    print(f'관점 {p}: 1위={top1[0]}({top1[1]:+.4f}) / 2위={top2[0]}({top2[1]:+.4f})')

=== 관점별 가장 큰 NDCG@10 기여 요소 (제거 시 가장 큰 하락) ===
관점 A: 1위=industry(+0.0102) / 2위=competency(+0.0017)
관점 B: 1위=quality_adjustment(+0.0080) / 2위=competency(+0.0049)
관점 C: 1위=role_semantic(+0.0000) / 2위=hard_skill(+0.0000)
관점 D: 1위=industry(+0.0056) / 2위=role_semantic(+0.0000)
관점 E: 1위=achievement(+0.0082) / 2위=industry(+0.0058)


In [6]:
# 요약 저장
summary = {
    'exp_id': 'exp-015',
    'title': 'Ablation Study — fusion 6요소 단일 제거',
    'date': '2026-05-18',
    'baseline_weights': SINGLE_WEIGHTS,
    'full_ndcg10_per_perspective': {p: float(full_row[p]) for p in 'ABCDE'},
    'full_ndcg10_mean': float(full_row['mean']),
    'feature_contribution_mean': {row.name: float(row['mean']) for _, row in contrib.iterrows()},
    'most_important_feature': str(contrib.index[0]),
    'least_important_feature': str(contrib.index[-1]),
    'note': '양의 기여 = 빼면 NDCG↓, 음의 기여 = 빼면 NDCG↑ (노이즈)',
}
with open(OUT_DIR / 'exp015_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "exp_id": "exp-015",
  "title": "Ablation Study — fusion 6요소 단일 제거",
  "date": "2026-05-18",
  "baseline_weights": {
    "role_semantic": 0.35,
    "hard_skill": 0.2,
    "competency": 0.15,
    "achievement": 0.1,
    "industry": 0.1,
    "quality_adjustment": 0.1
  },
  "full_ndcg10_per_perspective": {
    "A": 0.9793575633440395,
    "B": 0.9367249910106381,
    "C": 0.9226709526337592,
    "D": 0.9564046605799732,
    "E": 0.9613632205124331
  },
  "full_ndcg10_mean": 0.9513042776161684,
  "feature_contribution_mean": {
    "industry": 0.0030716390814052863,
    "quality_adjustment": 0.0017821229365567737,
    "role_semantic": 0.00029622012893282257,
    "competency": -0.0004441833593453737,
    "achievement": -0.0008501416007726803,
    "hard_skill": -0.003416214553669805
  },
  "most_important_feature": "industry",
  "least_important_feature": "hard_skill",
  "note": "양의 기여 = 빼면 NDCG↓, 음의 기여 = 빼면 NDCG↑ (노이즈)"
}
